In [1]:
# torch.manual_seed(0): identical tensors on every rerun — a notebook that
# explains specific shapes and values must be reproducible.
# The ROOT walk: jupyter's cwd depends on how the session was launched, so we
# search upward for al_haytham_llm/config.py and prepend the repo root to sys.path.

import math
import pathlib
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

def p(*text):
    print(*text)

ROOT = pathlib.Path.cwd()

while not (ROOT / "al_haytham_llm" / "config.py").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
    
sys.path.insert(0,str(ROOT))

torch.manual_seed(0)
print(ROOT)

c:\Mo2zy\Projects\Al-Haytham


In [2]:
# Toy hyper-parameters vs the real defaults in config.py —
# d_model 6 (vs 768), n_heads 2 (vs 12) -> per-head width d_k = 3 (vs 64),
# vocab 32, seq_len 16. Small enough to print every number, big enough to
# exercise all the reshape/transpose logic downstream.
# x stands in for embedding output: (batch, seq_len, d_model).

from al_haytham_llm.models.attention import MultiHeadAttention
from al_haytham_llm.models.feedforward import FeedForward
from al_haytham_llm.config import ModelConfig

TOKENS = ["the","cat",'sat','on','mat']

config =  ModelConfig(
 vocab_size=32,
 seq_len=16,
 d_model= 8,
 n_heads= 2,
 n_layers=2,
 qkv_bias=False,
 dropout =0.1
)

batch = 1
seq_len = len(TOKENS)

x = torch.rand(batch,seq_len,config.d_model)

print (config)

print(x.shape,x)



ModelConfig(vocab_size=32, seq_len=16, d_model=8, d_ff=3072, n_heads=2, n_layers=2, qkv_bias=False, dropout=0.1, pad_id=0, tie_weights=True)
torch.Size([1, 5, 8]) tensor([[[0.4963, 0.7682, 0.0885, 0.1320, 0.3074, 0.6341, 0.4901, 0.8964],
         [0.4556, 0.6323, 0.3489, 0.4017, 0.0223, 0.1689, 0.2939, 0.5185],
         [0.6977, 0.8000, 0.1610, 0.2823, 0.6816, 0.9152, 0.3971, 0.8742],
         [0.4194, 0.5529, 0.9527, 0.0362, 0.1852, 0.3734, 0.3051, 0.9320],
         [0.1759, 0.2698, 0.1507, 0.0317, 0.2081, 0.9298, 0.7231, 0.7423]]])


In [3]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

layernorm_head = nn.LayerNorm(config.d_model)

language_model_head = nn.Linear(
    config.d_model,
    config.vocab_size,
    bias=False
)

normed = layernorm_head(x)

logits = language_model_head(normed)

# Predictions for the second token: "cat"
token_logits = logits[0, 1]

p(f"Logits for '{TOKENS[1]}': {token_logits}")

probs = F.softmax(logits, dim=-1)

token_probs = probs[0, 1]
p(f"Probabilities after '{TOKENS[1]}': {token_probs}")

top_k = torch.topk(token_probs, k=5)

p(f"Top 5 predictions: {top_k}")

Logits for 'cat': tensor([ 0.4126, -0.5962, -0.6906, -0.7136,  0.5680,  0.0114, -0.8886, -0.7124,
        -0.0374,  0.4874,  0.8677, -0.1654, -0.9647,  0.0036, -0.0616,  0.0601,
        -0.6663, -0.3357,  0.7267,  0.1628,  0.7348,  0.1637, -0.8635,  0.6116,
         0.5513,  0.1410, -0.5457,  0.3400, -0.8015, -0.3690, -0.5462, -0.3961],
       grad_fn=<SelectBackward0>)
Probabilities after 'cat': tensor([0.0455, 0.0166, 0.0151, 0.0148, 0.0532, 0.0305, 0.0124, 0.0148, 0.0290,
        0.0490, 0.0717, 0.0255, 0.0115, 0.0302, 0.0283, 0.0320, 0.0155, 0.0215,
        0.0623, 0.0354, 0.0628, 0.0355, 0.0127, 0.0555, 0.0523, 0.0347, 0.0175,
        0.0423, 0.0135, 0.0208, 0.0174, 0.0203], grad_fn=<SelectBackward0>)
Top 5 predictions: torch.return_types.topk(
values=tensor([0.0717, 0.0628, 0.0623, 0.0555, 0.0532], grad_fn=<TopkBackward0>),
indices=tensor([10, 20, 18, 23,  4]))


In [4]:
from al_haytham_llm.models.blocks_of_transformer import TransformerBlock

blocks = nn.ModuleList(
    TransformerBlock(config)
    for i in range(config.n_layers)
)

blocks

ModuleList(
  (0-1): 2 x TransformerBlock(
    (attention): MultiHeadAttention(
      (query_projection): Linear(in_features=8, out_features=8, bias=False)
      (key_projection): Linear(in_features=8, out_features=8, bias=False)
      (value_projection): Linear(in_features=8, out_features=8, bias=False)
      (output_projection): Linear(in_features=8, out_features=8, bias=True)
    )
    (feedforward): FeedForward(
      (expansion_layer): Linear(in_features=8, out_features=3072, bias=True)
      (activation_function): GELU(approximate='tanh')
      (projection_layer): Linear(in_features=3072, out_features=8, bias=True)
    )
    (layer_norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (layer_norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout_layer): Dropout(p=0.1, inplace=False)
  )
)

## Weight Tying

In [5]:
from al_haytham_llm.models.embeddings import InputEmbedding

embed = InputEmbedding(config)

language_model_head = nn.Linear(
    config.d_model,
    config.vocab_size,
    bias=False
)

p("language_model_head shape is : ",language_model_head.weight.shape)
p("embed_model shape is : ",embed.token_embedding.weight.shape)



language_model_head = embed.token_embedding.weight # so now both using the same weight and memory (efficeint)

language_model_head shape is :  torch.Size([32, 8])
embed_model shape is :  torch.Size([32, 8])


## Initialisation


In [6]:
def _init_weights(module):
    """GPT-2's initialisation: normal(0, 0.02), zeroed biases.

        PyTorch's Linear default depends on fan-in, which leaves a 768-wide
        model's activations larger than GPT-2 assumes and makes the early
        steps of training less stable.
    """

    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weights, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weights, mean=0.0, std=0.02)

def _scale_residual_projections(blocks, config):

    scale = math.sqrt( 2 *  config.n_layers)
    for block in blocks:
        with torch.no_grad():
            block.attention.output_projection.weight.div_(scale)
            block.feedforward.projection_layer.weight.div_(scale)

### Flatten the logits to be from (B,T,dm) to (B*T,) so that B batches of T tokens

In [7]:
sentence = torch.tensor([[5, 9, 14, 3, 21, 7]])

inputs = sentence[:, 0:-1]
targets = sentence[:, 1:]

print("sentence:", sentence.tolist()[0])
print("inputs  :", inputs.tolist()[0])
print("targets :", targets.tolist()[0])

print()
print(f"{'position':>9}{'reads':>8}{'must predict':>14}")
for t in range(inputs.size(1)):
    print(f"{t:>9}{inputs[0, t].item():>8}{targets[0, t].item():>14}")

sentence: [5, 9, 14, 3, 21, 7]
inputs  : [5, 9, 14, 3, 21]
targets : [9, 14, 3, 21, 7]

 position   reads  must predict
        0       5             9
        1       9            14
        2      14             3
        3       3            21
        4      21             7


In [8]:
ln_final = nn.LayerNorm(config.d_model)
lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)

normed = ln_final(x)
logits = lm_head(normed)

print("logits          :", tuple(logits.shape)) # (B, T, vocab_size) 
print("targets          :", tuple(targets.shape)) # (B, T)

# # flatten (B*T, vocab_size) and (B*T,) for loss computation

# flatten_logits = logits.reshape(-1 , logits.size(-1))
# print("flatten_logits          :", tuple(flatten_logits.shape))

# flatten_targets = targets.reshape(-1)
# print("flatten_targets          :", tuple(flatten_targets.shape))

# # === Example:

# # logits       (B, T, V)   = (2, 5, 50257)   ← scores for every token at every position
# # targets      (B, T)      = (2, 5)          ← the correct token id at each position

# # logits.reshape(-1, lg.size(-1))  →  (10, 50257)   ← 10 separate predictions
# # targets.reshape(-1)              →  (10,)         ← 10 correct answers

# # ===

# loss = F.cross_entropy( flatten_logits, flatten_targets )

loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

logits          : (1, 5, 32)
targets          : (1, 5)


In [9]:
loss

tensor(4.1671, grad_fn=<NllLossBackward0>)

In [10]:
# هنا انا لما مجدد ان السيكونس لينث بيبقى 12 ف لو دخلي اقل بملاها ب رقم ان محدد و وقت حساب اللوس بتجاهله
# fill to 12
sentence = torch.tensor([[5, 9, 14, 3, 21, 7, config.pad_id, config.pad_id, config.pad_id,
                                              config.pad_id, config.pad_id, config.pad_id]])

loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                       ignore_index=config.pad_id)

___
# Calc my GPT Params

In [ ]:
from al_haytham_llm.models.gpt import GPTModel

my_gpt = GPTModel(config.tiny())

print(f"Total Number of Parameters: {my_gpt.num_parameters():,}")
p("Non-Embedding Parameters:", f"{my_gpt.num_parameters(non_embedding=True):,}")

Total Number of Parameters: 5,385,344
Non-Embedding Parameters: 3,304,576
